# Proposed seasonal-naive comparison on 7-day Development

Repeat the previous observed 24-hour pattern for entities 1001 and 1871 using the same input, 90 origins, and label cutoff as the three-seed adaptive run. This fallback has not been approved as the official #22 baseline. Do not use Final/test or choose parameters from these results. The runtime is set to T4 as requested, but the pinned CPU PyTorch package does not use the GPU.

Each Run all creates a new run ID and Drive folder without overwriting previous checkpoints or reports.


In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

CODE_SHA = "1d33418e8953fcf9bff8f360c802a3daeb45878c"
BUNDLE_NAME = "colab-code-1d33418-seasonal-20261023.bundle"
BUNDLE_SHA = "29a6aa4959bb574e3a1d7a96b5ea3745a980fd6e4e199a6e178d6b5f3e26df14"
INPUT_FILE_ID = "17pfYqnnSq0Uao2zDNdb8h7GAYfotu_w4"
INPUT_MANIFEST_SHA = "fd1b5d1767e94a3e5c6f62447d446e1f13859b823c5fe14c51b5055d8e67169c"
PRIOR_RUN_ID = "hung-urbanev-7d-dev-multiseed-20261023T161534600069Z"
PRIOR_CODE_SHA = "85e45606789eaff7e2e929eb7a7ee353a4ee49f9"
DRIVE_RUN_PARENT = Path("/content/drive/MyDrive/hung-sprint2-runs")
RUN_ID = "hung-urbanev-7d-seasonal-comparison-" + datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)
TARGETS = ("1001", "1871")
EXPECTED_SCORED = {"1001": 52, "1871": 38}
print("ma_luot_chay", RUN_ID)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_drive_run(folder, expected_code_sha):
    folder = Path(folder)
    marker = folder / "handoff_complete.json"
    if not marker.is_file():
        raise ValueError(f"Missing Drive handoff marker: {folder}")
    record = json.loads(marker.read_text(encoding="utf-8"))
    if (
        record.get("schema") != "hung-sprint2-drive-handoff-v1"
        or record.get("code_sha") != expected_code_sha
    ):
        raise ValueError("Drive handoff schema/code mismatch")
    actual = {
        p.relative_to(folder).as_posix() for p in folder.rglob("*") if p.is_file() and p != marker
    }
    if actual != set(record["files"]):
        raise ValueError("Drive handoff file set mismatch")
    for name, expected in record["files"].items():
        if sha256_file(folder / name) != expected:
            raise ValueError(f"Drive handoff file hash mismatch: {name}")
    return record


def publish_run(local_folder, drive_folder):
    local_folder = Path(local_folder)
    drive_folder = Path(drive_folder)
    drive_folder.parent.mkdir(parents=True, exist_ok=True)
    if drive_folder.exists():
        raise FileExistsError(drive_folder)
    shutil.copytree(local_folder, drive_folder)
    files = {}
    for original in local_folder.rglob("*"):
        if original.is_file():
            name = original.relative_to(local_folder).as_posix()
            digest = sha256_file(original)
            if digest != sha256_file(drive_folder / name):
                raise ValueError(f"Drive copy hash mismatch: {name}")
            files[name] = digest
    marker = {"schema": "hung-sprint2-drive-handoff-v1", "code_sha": CODE_SHA, "files": files}
    with (drive_folder / "handoff_complete.json").open("x", encoding="utf-8") as stream:
        json.dump(marker, stream, indent=2, sort_keys=True)
        stream.write("\n")
    verify_drive_run(drive_folder, CODE_SHA)
    return drive_folder

In [ ]:
from importlib.metadata import version

repo = Path("/content/hung-sprint2-code-" + RUN_ID)
bundle = Path("/content/drive/MyDrive") / BUNDLE_NAME
if repo.exists() or not bundle.is_file():
    raise FileExistsError("Code destination exists or bundle is missing")
if sha256_file(bundle) != BUNDLE_SHA:
    raise ValueError("Code bundle SHA-256 mismatch")
subprocess.run(["git", "clone", str(bundle), str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", CODE_SHA], check=True)
actual_sha = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()
if actual_sha != CODE_SHA:
    raise ValueError("Checked-out code commit mismatch")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements-dev.txt")], check=True
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--index-url",
        "https://download.pytorch.org/whl/cpu",
        "-r",
        str(repo / "requirements-model.txt"),
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "gdown==5.2.0"], check=True)
if version("torch") != "2.8.0+cpu" or version("gdown") != "5.2.0":
    raise RuntimeError("Unexpected pinned dependency version")
audit = subprocess.run([sys.executable, "-m", "pip", "check"], text=True, capture_output=True)
if audit.returncode:
    print("canh_bao_goi_colab:", audit.stdout.strip(), audit.stderr.strip())
print("commit_code", actual_sha, "sha256_archive_code", BUNDLE_SHA)

In [ ]:
archive = Path("/content/inputs-urbanev-SZH-v1.zip")
input_root = Path("/content/hung-sprint2-data") / RUN_ID
if input_root.exists():
    raise FileExistsError(input_root)
if not archive.exists():
    import gdown

    downloaded = gdown.download(id=INPUT_FILE_ID, output=str(archive), quiet=False)
    if downloaded != str(archive):
        raise RuntimeError("Input download failed")
subprocess.run(
    [
        sys.executable,
        "-m",
        "experiments.hung_sprint2.cli",
        "verify-input-zip",
        "--archive",
        str(archive),
        "--package",
        archive.name,
    ],
    cwd=repo,
    check=True,
)
with zipfile.ZipFile(archive) as package:
    names = package.namelist()
    prefix = "data/inputs/issue18-input-handoff-v1-20261019/urbanev-SZH-"
    if len(names) != len(set(names)) or any(
        not name.startswith(prefix) or Path(name).is_absolute() or ".." in Path(name).parts
        for name in names
    ):
        raise ValueError("Unexpected archive entry")
    input_root.mkdir(parents=True)
    package.extractall(input_root)
run = input_root / "data/inputs/issue18-input-handoff-v1-20261019/urbanev-SZH-7d"
if sha256_file(run / "manifest.json") != INPUT_MANIFEST_SHA:
    raise ValueError("Accepted 7-day input manifest mismatch")
print("bo_dau_vao", run.name, INPUT_MANIFEST_SHA)

In [ ]:
run_root = Path("/content/hung-sprint2-runs") / RUN_ID
run_root.mkdir(parents=True, exist_ok=False)
prior = DRIVE_RUN_PARENT / PRIOR_RUN_ID
prior_marker = verify_drive_run(prior, PRIOR_CODE_SHA)
prior_summary = json.loads((prior / "development_summary.json").read_text(encoding="utf-8"))
if (
    prior_summary["run_id"] != PRIOR_RUN_ID
    or prior_summary["input_manifest_sha256"] != INPUT_MANIFEST_SHA
):
    raise ValueError("Prior Development summary identity mismatch")
comparisons = []
for entity_id in TARGETS:
    output = run_root / f"entity-{entity_id}"
    command = [
        sys.executable,
        "-m",
        "experiments.hung_sprint2.seasonal_naive_replay",
        "--input-run",
        str(run),
        "--manifest-sha256",
        INPUT_MANIFEST_SHA,
        "--entity-id",
        entity_id,
        "--max-origins",
        "90",
        "--output",
        str(output),
    ]
    print("lenh_chay:", " ".join(command))
    subprocess.run(command, cwd=repo, check=True)
    baseline = json.loads((output / "run_manifest.json").read_text(encoding="utf-8"))
    if (
        baseline["coverage"]["scored"] != EXPECTED_SCORED[entity_id]
        or baseline["metrics_kwh"] is None
    ):
        raise ValueError("Baseline coverage differs from expected Development run")
    baseline_scored = set(baseline["scored_origins"])
    if len(baseline_scored) != EXPECTED_SCORED[entity_id]:
        raise ValueError("Baseline scored-origin count mismatch")
    for source in prior_summary["results"]:
        if source["entity_id"] != entity_id:
            continue
        relative = f"seed-{source['seed']}/entity-{entity_id}/replay-90/run_manifest.json"
        old_manifest = prior / relative
        if sha256_file(old_manifest) != prior_marker["files"][relative]:
            raise ValueError("Prior replay manifest hash mismatch")
        old = json.loads(old_manifest.read_text(encoding="utf-8"))
        old_scored = {
            event.split(":", 1)[1]
            for event in old["event_evidence"]["events"]
            if event.startswith("score:")
        }
        if old_scored != baseline_scored or len(old_scored) != source["coverage"]["scored"]:
            raise ValueError("Scored origin sets differ between model and baseline")
        model_mae = source["metrics_kwh"]["live"]["mae_kwh"]
        baseline_mae = baseline["metrics_kwh"]["mae_kwh"]
        comparisons.append(
            {
                "entity_id": entity_id,
                "seed": source["seed"],
                "scored_days": len(old_scored),
                "model_mae_kwh": model_mae,
                "proposed_seasonal_mae_kwh": baseline_mae,
                "model_minus_seasonal_mae_kwh": model_mae - baseline_mae,
            }
        )
    print("Development", entity_id, baseline["coverage"], baseline["metrics_kwh"])
if len(comparisons) != 6:
    raise ValueError("Expected six paired model-baseline comparisons")
with (run_root / "comparison_summary.json").open("x", encoding="utf-8") as stream:
    json.dump(
        {
            "run_id": RUN_ID,
            "code_sha": CODE_SHA,
            "input_manifest_sha256": INPUT_MANIFEST_SHA,
            "prior_run_id": PRIOR_RUN_ID,
            "baseline_status": "proposed_not_accepted",
            "comparisons": comparisons,
        },
        stream,
        ensure_ascii=False,
        indent=2,
    )
print("hoan_tat_so_sanh", len(comparisons))

In [ ]:
drive_run = publish_run(run_root, DRIVE_RUN_PARENT / RUN_ID)
print("thu_muc_drive", drive_run)
print("so_luot_so_sanh", len(comparisons))